In [2]:
import random
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import networkit as nk
import networkx as nx
import numpy as np
from networkit.embedding import Node2Vec
from sklearn.metrics import f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
nk.engineering.setNumberOfThreads(1)

## Networkit documentation
https://networkit.github.io/dev-docs/python_api/modules.html

## Loading network

In [ ]:
data_path = Path('./citeseer/citeseer.adjlist')
G = ...

## Sparsification

sparsyfikacja jest jednym z typów redukcji grafu, która wybiera istotne węzły i krawędzie, odrzucając pozostałe. W trakcie działania procesu dla danego grafu $G = (A, X)$,  model sparsyfikacji grafu wybiera istniejące węzły lub krawędzie z grafu $G$, tworząc $G' = (A', X')$. Innymi słowy, elementy $A'$ lub $X'$ są podzbiorem elementów w $A$ lub $X$.

In [ ]:
edge_ratio = 0.5

## Link prediction

celem zadania jest przewidywanie,
czy pomiędzy daną parą węzłów powinna istnieć krawędź.
Problem ten jest formułowany jako zadanie klasyfikacji binarnej,
gdzie:
- klasa pozytywna odpowiada istniejącym krawędziom,
- klasa negatywna odpowiada parom węzłów bez połączenia.

W naszym przypadku celem modelu jest odtworzenie struktury
zredukowanego grafu na podstawie dostępnych danych.

### Negative sampling

proces polegający na losowaniu par węzłów,
które **nie są połączone krawędzią** w grafie.

Jest to kluczowy krok w zadaniu link prediction, ponieważ:
- graf nie zawiera jawnych przykładów negatywnych,
- model klasyfikacyjny wymaga danych obu klas,
- losowanie negatywnych krawędzi umożliwia zbalansowanie zbioru danych.

### Cechy krawędzi (edge features)

w tym ćwiczeniu cechy krawędzi są budowane na podstawie
**embeddingów węzłów uzyskanych metodą Node2Vec**.

Dla każdej krawędzi (u, v):
- pobierane są embeddingi węzłów u i v,
- łączone są w jeden wektor cech.


In [ ]:
def sample_negative_edges(
    G: nk.Graph,
    num_samples: int,
    forbidden_edges: set[tuple[int, int]] | None = None,
) -> np.ndarray:
    """
    Funkcja powinna losować "negatywne krawędzie" w grafie, czyli takie, które nie istnieją w grafie G.

    Parametry
    G : nk.Graph
        Graf wejściowy
    num_samples : int
        Liczba negatywnych krawędzi do wylosowania
    forbidden_edges : set[tuple[int,int]] lub None
        Zbiór krawędzi, których NIE można wylosować (np. istniejących w grafie)

    Wskazówki:
    - Pomiń pętle (u==v)
    - Normalizuj pary dla grafu nieskierowanego (u<=v)
    - Pomijaj krawędzie istniejące i zakazane
    - Wylosowane krawędzie mogą być zwracane w formie np. numpy array
    """
    pass


def get_edge_features(
    edge_list: Iterable[tuple[int, int]],
    emb: np.ndarray,
) -> np.ndarray:
    """
    Funkcja ma tworzyć wektory cech dla krawędzi na podstawie embeddingów węzłów.

    Parametry
    edge_list : Iterable[tuple[int,int]]
        Lista krawędzi, np. [(u1,v1), (u2,v2), ...]
    emb : np.ndarray, shape=(n_nodes, embedding_dim)
        Macierz embeddingów węzłów (np. wynik Node2Vec)

    Output
    np.ndarray, shape=(len(edge_list), embedding_dim)
        Każdy wiersz to wektor cech krawędzi, np. iloczyn Hadamarda
        wektorów węzłów u i v

    Wskazówki:
    - Dla każdej krawędzi (u,v) pobierz emb[u] i emb[v]
    - Połącz je w jedną cechę (np. iloczyn Hadamarda)
    - Zbierz wszystkie krawędzie w jedną tablicę numpy
    """
    pass

def preprocess_graph(G: nk.Graph) -> nk.Graph:
    """
    Przygotuj graf do Node2Vec

    Wskazówki:
    1. Konwertuj NetworKit -> NetworkX
    2. Usuń izolowane węzły
    3. Przypisz węzłom nowe ID 0..n-1
    4. Konwertuj z powrotem NetworkX -> NetworKit
    """
    pass

In [ ]:
reduced_data_path = Path('./citeseer/0_9_citeseer.adjlist')
reduced_G = ...

In [ ]:
preprocessed_G = preprocess_graph(G)
preprocessed_reduced_G  = preprocess_graph(reduced_G)

orig_edges = list(preprocessed_G.iterEdges())
reduced_edges = list(preprocessed_reduced_G.iterEdges())

removed_edges = list(set(orig_edges) - set(reduced_edges))

print("Original edges:", len(orig_edges))
print("Reduced edges:", len(reduced_edges))
print("Removed edges (to predict):", len(removed_edges))

### Node2Vec
model uczenia reprezentacji węzłów w grafach, która odwzorowuje węzły na wektory w przestrzeni ciągłej, zachowując strukturę sieci. Kluczowym elementem node2vec jest *random walk*. Formalnie, dla danego węzła źródłowego $u$ symulujemy random walk o stałej długości $l$. Niech $c_i$ oznacza i‑ty węzeł w ścieżce, zaczynając od $c_0 = u$. Węzły $c_i$ są generowane zgodnie z rozkładem:

$$
P(c_i = x \mid c_{i-1} = v) = 
\begin{cases}
\frac{\pi_{v x}}{Z}, & \text{if} (v,x) \in E, \\
0, & \text{otherwise},
\end{cases}
$$

gdzie \($\pi_{v x}$\) to nienormalizowane prawdopodobieństwo przejścia między węzłami $v$ i $x$, a \($Z$\) jest stałą normalizującą.

Networkit documentation: https://networkit.github.io/dev-docs/python_api/embedding.html


In [ ]:
node2vec = Node2Vec(
    G=preprocessed_reduced_G,
    P=...,
    Q=...,
    L=...,
    N=...,
    D=...,
)
node2vec.run()
emb_matrix = np.array(node2vec.getFeatures())
print("Embeddings shape:", emb_matrix.shape)

### Budowa zbioru treningowego

W tym zdaniu zbiór treningowy jest budowany poprzez łączenie
próbek pozytywnych i negatywnych.

W kodzie:
- `sample_negative_edges` - wykorzystujemy do losowania par węzłów,
    pomiędzy którymi nie istnieje krawędź w grafie.
- `edge_features`
- `np.vstack` łączy macierze cech:
  - wiersze odpowiadają poszczególnym krawędziom,
  - najpierw krawędzie istniejące, potem negatywne,
- `np.hstack` łączy wektory etykiet:
  - `1` dla krawędzi istniejących,
  - `0` dla krawędzi negatywnych.

Ten sam proces jest wykorzystywny dla sieci powstałej po redukcji służącej jako zbiór treningowej oraz elementom sieci usuniętym w trakcie redukcji służącej jako zbiór testowy.

In [ ]:
forbidden = set(orig_edges)
neg_edges = sample_negative_edges(preprocessed_reduced_G, len(reduced_edges), forbidden_edges=forbidden)

X_train = np.vstack([get_edge_features(reduced_edges, emb_matrix),
                     get_edge_features(neg_edges, emb_matrix)])
y_train = np.hstack([np.ones(len(reduced_edges)), np.zeros(len(neg_edges))])

neg_test = sample_negative_edges(preprocessed_reduced_G, len(removed_edges), forbidden_edges=forbidden)
X_test = np.vstack([get_edge_features(removed_edges, emb_matrix),
                    get_edge_features(neg_test, emb_matrix)])
y_test = np.hstack([np.ones(len(removed_edges)), np.zeros(len(neg_test))])

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

clf = ...
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

In [ ]:
print("Test F1:", f1_score(y_test, y_pred, average='macro'))